## Imports and project paths

In [1]:
# ============================================================
# 1. Imports and project paths
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


BASE_DIR = Path.cwd().parent

OUTPUT_DIR = BASE_DIR / "outputs"

FORECAST_PATH = (
    OUTPUT_DIR
    / "forecast_results.csv"
)

RISK_PATH = (
    OUTPUT_DIR
    / "risk_results.csv"
)

ANOMALY_PATH = (
    OUTPUT_DIR
    / "anomaly_results.csv"
)

EXPLAINABILITY_PATH = (
    OUTPUT_DIR
    / "explainability_results.csv"
)


print("=" * 72)
print("EARLYSIGNAL AI — EARLY WARNING PRIORITY ENGINE")
print("=" * 72)

print("\nBase directory:")
print(BASE_DIR)

print("\nRequired outputs:")

for path in [
    FORECAST_PATH,
    RISK_PATH,
    ANOMALY_PATH,
    EXPLAINABILITY_PATH
]:
    print(
        f" - {path.name}: {path.exists()}"
    )

EARLYSIGNAL AI — EARLY WARNING PRIORITY ENGINE

Base directory:
C:\Users\Ezekiel Mbaya\myenv\EarlySignal_AI

Required outputs:
 - forecast_results.csv: True
 - risk_results.csv: True
 - anomaly_results.csv: True
 - explainability_results.csv: True


## Load model outputs

In [2]:
# ============================================================
# 2. Load model outputs
# ============================================================

forecast_results = pd.read_csv(
    FORECAST_PATH,
    parse_dates=["reporting_date"]
)

risk_results = pd.read_csv(
    RISK_PATH,
    parse_dates=["reporting_date"]
)

anomaly_results = pd.read_csv(
    ANOMALY_PATH,
    parse_dates=["reporting_date"]
)

explainability_results = pd.read_csv(
    EXPLAINABILITY_PATH,
    parse_dates=["reporting_date"]
)


print("=" * 72)
print("EARLY WARNING INPUT DATASETS")
print("=" * 72)

print(
    "\nForecast results:",
    forecast_results.shape
)

print(
    "Risk results:",
    risk_results.shape
)

print(
    "Anomaly results:",
    anomaly_results.shape
)

print(
    "Explainability results:",
    explainability_results.shape
)

print("\nDate ranges:")

for name, frame in {
    "Forecast": forecast_results,
    "Risk": risk_results,
    "Anomaly": anomaly_results,
    "Explainability": explainability_results
}.items():

    print(
        f"{name}: "
        f"{frame['reporting_date'].min().date()} "
        f"to "
        f"{frame['reporting_date'].max().date()}"
    )

EARLY WARNING INPUT DATASETS

Forecast results: (1440, 12)
Risk results: (1680, 14)
Anomaly results: (1680, 15)
Explainability results: (1680, 15)

Date ranges:
Forecast: 2026-02-01 to 2026-07-01
Risk: 2026-02-01 to 2026-08-01
Anomaly: 2026-02-01 to 2026-08-01
Explainability: 2026-02-01 to 2026-08-01


## Validate unique record IDs

In [3]:
# ============================================================
# 3. Validate integration keys
# ============================================================

integration_frames = {
    "Forecast":
        forecast_results,

    "Risk":
        risk_results,

    "Anomaly":
        anomaly_results,

    "Explainability":
        explainability_results
}


print("=" * 72)
print("INTEGRATION KEY VALIDATION")
print("=" * 72)

for name, frame in integration_frames.items():

    duplicate_ids = int(
        frame[
            "record_id"
        ].duplicated().sum()
    )

    missing_ids = int(
        frame[
            "record_id"
        ].isna().sum()
    )

    print(
        f"\n{name}:"
    )

    print(
        " Rows:",
        len(frame)
    )

    print(
        " Duplicate record IDs:",
        duplicate_ids
    )

    print(
        " Missing record IDs:",
        missing_ids
    )

INTEGRATION KEY VALIDATION

Forecast:
 Rows: 1440
 Duplicate record IDs: 0
 Missing record IDs: 0

Risk:
 Rows: 1680
 Duplicate record IDs: 0
 Missing record IDs: 0

Anomaly:
 Rows: 1680
 Duplicate record IDs: 0
 Missing record IDs: 0

Explainability:
 Rows: 1680
 Duplicate record IDs: 0
 Missing record IDs: 0


## Audit cross-model record overlap

In [4]:
# ============================================================
# 4. Audit cross-model record overlap
# ============================================================

forecast_ids = set(
    forecast_results[
        "record_id"
    ]
)

risk_ids = set(
    risk_results[
        "record_id"
    ]
)

anomaly_ids = set(
    anomaly_results[
        "record_id"
    ]
)

explainability_ids = set(
    explainability_results[
        "record_id"
    ]
)

common_all_ids = (
    forecast_ids
    & risk_ids
    & anomaly_ids
    & explainability_ids
)

risk_anomaly_explain_ids = (
    risk_ids
    & anomaly_ids
    & explainability_ids
)


print("=" * 72)
print("CROSS-MODEL RECORD OVERLAP")
print("=" * 72)

print(
    "\nForecast records:",
    len(forecast_ids)
)

print(
    "Risk records:",
    len(risk_ids)
)

print(
    "Anomaly records:",
    len(anomaly_ids)
)

print(
    "Explainability records:",
    len(explainability_ids)
)

print(
    "\nRecords available across all four components:",
    len(common_all_ids)
)

print(
    "Records available across risk + anomaly + explainability:",
    len(risk_anomaly_explain_ids)
)

print(
    "\nRisk records without a held-out forecast:",
    len(
        risk_ids
        - forecast_ids
    )
)

CROSS-MODEL RECORD OVERLAP

Forecast records: 1440
Risk records: 1680
Anomaly records: 1680
Explainability records: 1680

Records available across all four components: 1440
Records available across risk + anomaly + explainability: 1680

Risk records without a held-out forecast: 240


## Audit the forecast gap

In [5]:
# ============================================================
# 5. Audit records without forecast output
# ============================================================

missing_forecast_ids = (
    risk_ids
    - forecast_ids
)

missing_forecast_records = (
    risk_results[
        risk_results[
            "record_id"
        ].isin(
            missing_forecast_ids
        )
    ]
    .sort_values(
        "reporting_date"
    )
)


print("=" * 72)
print("FORECAST COVERAGE AUDIT")
print("=" * 72)

print(
    "\nRecords without forecast output:",
    len(missing_forecast_records)
)

if len(missing_forecast_records) > 0:

    print(
        "Date range:",
        missing_forecast_records[
            "reporting_date"
        ].min().date(),
        "to",
        missing_forecast_records[
            "reporting_date"
        ].max().date()
    )

    print(
        "\nRecords by reporting month:"
    )

    print(
        missing_forecast_records[
            "reporting_date"
        ].value_counts().sort_index()
    )

FORECAST COVERAGE AUDIT

Records without forecast output: 240
Date range: 2026-08-01 to 2026-08-01

Records by reporting month:
reporting_date
2026-08-01    240
Name: count, dtype: int64


## Verify identity fields across outputs

In [6]:
# ============================================================
# 6. Verify record identity consistency
# ============================================================

identity_columns = [
    "record_id",
    "reporting_date",
    "programme_name",
    "sector",
    "state",
    "lga",
    "community"
]

risk_identity = (
    risk_results[
        identity_columns
    ]
    .sort_values(
        "record_id"
    )
    .reset_index(drop=True)
)

anomaly_identity = (
    anomaly_results[
        identity_columns
    ]
    .sort_values(
        "record_id"
    )
    .reset_index(drop=True)
)

explain_identity = (
    explainability_results[
        identity_columns
    ]
    .sort_values(
        "record_id"
    )
    .reset_index(drop=True)
)


risk_anomaly_identity_match = (
    risk_identity.equals(
        anomaly_identity
    )
)

risk_explain_identity_match = (
    risk_identity.equals(
        explain_identity
    )
)


print("=" * 72)
print("RECORD IDENTITY CONSISTENCY")
print("=" * 72)

print(
    "\nRisk vs Anomaly identity match:",
    risk_anomaly_identity_match
)

print(
    "Risk vs Explainability identity match:",
    risk_explain_identity_match
)

print(
    "\nIdentity audit:",
    (
        "PASSED"
        if (
            risk_anomaly_identity_match
            and risk_explain_identity_match
        )
        else "CHECK REQUIRED"
    )
)

RECORD IDENTITY CONSISTENCY

Risk vs Anomaly identity match: True
Risk vs Explainability identity match: True

Identity audit: PASSED


## Build the master EarlySignal table

In [7]:
# ============================================================
# 7. Build master EarlySignal integration table
# ============================================================

master = risk_results[
    [
        "record_id",
        "reporting_date",
        "programme_name",
        "sector",
        "state",
        "lga",
        "community",
        "achievement_rate",
        "risk_label",
        "predicted_risk",
        "probability_high"
    ]
].copy()


# Add anomaly outputs
master = master.merge(
    anomaly_results[
        [
            "record_id",
            "predicted_anomaly",
            "anomaly_score"
        ]
    ],
    on="record_id",
    how="left",
    validate="one_to_one"
)


# Add explainability outputs
master = master.merge(
    explainability_results[
        [
            "record_id",
            "primary_driver",
            "secondary_driver",
            "tertiary_driver",
            "strongest_protective_signal",
            "recommended_investigation",
            "decision_use"
        ]
    ],
    on="record_id",
    how="left",
    validate="one_to_one"
)


# Add forecast outputs
master = master.merge(
    forecast_results[
        [
            "record_id",
            "predicted_next_achievement_rate",
            "forecast_status"
        ]
    ],
    on="record_id",
    how="left",
    validate="one_to_one"
)


master["forecast_available"] = (
    master[
        "predicted_next_achievement_rate"
    ].notna()
)


print("=" * 74)
print("MASTER EARLYSIGNAL INTEGRATION TABLE")
print("=" * 74)

print(
    "\nRows:",
    len(master)
)

print(
    "Columns:",
    len(master.columns)
)

print(
    "Duplicate record IDs:",
    int(
        master[
            "record_id"
        ].duplicated().sum()
    )
)

print(
    "\nForecast available:",
    int(
        master[
            "forecast_available"
        ].sum()
    )
)

print(
    "Forecast unavailable:",
    int(
        (~master[
            "forecast_available"
        ]).sum()
    )
)

print(
    "\nMissing anomaly outputs:",
    int(
        master[
            "predicted_anomaly"
        ].isna().sum()
    )
)

print(
    "Missing explanations:",
    int(
        master[
            "primary_driver"
        ].isna().sum()
    )
)

MASTER EARLYSIGNAL INTEGRATION TABLE

Rows: 1680
Columns: 22
Duplicate record IDs: 0

Forecast available: 1440
Forecast unavailable: 240

Missing anomaly outputs: 0
Missing explanations: 0


## Verify forecast coverage flag

In [8]:
# ============================================================
# 8. Verify forecast availability logic
# ============================================================

forecast_coverage = (
    master
    .groupby(
        "reporting_date"
    )
    .agg(
        records=(
            "record_id",
            "size"
        ),
        forecasts_available=(
            "forecast_available",
            "sum"
        )
    )
)

forecast_coverage[
    "forecasts_unavailable"
] = (
    forecast_coverage[
        "records"
    ]
    - forecast_coverage[
        "forecasts_available"
    ]
)


print("=" * 74)
print("FORECAST COVERAGE BY REPORTING MONTH")
print("=" * 74)

display(
    forecast_coverage
)

FORECAST COVERAGE BY REPORTING MONTH


,records,forecasts_available,forecasts_unavailable
reporting_date,,,
2026-02-01,240,240,0
2026-03-01,240,240,0
2026-04-01,240,240,0
2026-05-01,240,240,0
2026-06-01,240,240,0
2026-07-01,240,240,0
2026-08-01,240,0,240


## Define transparent component scores

In [9]:
# ============================================================
# 9. Define transparent component scores
# ============================================================

risk_score_map = {
    "Low": 0.0,
    "Medium": 50.0,
    "High": 100.0
}

forecast_score_map = {
    "On Track": 0.0,
    "Watch": 50.0,
    "Critical": 100.0
}


master[
    "risk_component_score"
] = (
    master[
        "predicted_risk"
    ].map(
        risk_score_map
    )
)


master[
    "forecast_component_score"
] = (
    master[
        "forecast_status"
    ].map(
        forecast_score_map
    )
)


master[
    "anomaly_component_score"
] = np.where(
    master[
        "predicted_anomaly"
    ] == 1,
    100.0,
    0.0
)


print("=" * 74)
print("EARLY WARNING COMPONENT SCORES")
print("=" * 74)

print("\nRisk component:")
print(
    master[
        "risk_component_score"
    ].value_counts().sort_index()
)

print("\nForecast component:")
print(
    master[
        "forecast_component_score"
    ].value_counts(
        dropna=False
    ).sort_index()
)

print("\nAnomaly component:")
print(
    master[
        "anomaly_component_score"
    ].value_counts().sort_index()
)

EARLY WARNING COMPONENT SCORES

Risk component:
risk_component_score
0.0      925
50.0     615
100.0    140
Name: count, dtype: int64

Forecast component:
forecast_component_score
0.0      574
50.0     784
100.0     82
NaN      240
Name: count, dtype: int64

Anomaly component:
anomaly_component_score
0.0      1646
100.0      34
Name: count, dtype: int64


## Define the Early Warning Score

In [10]:
# ============================================================
# 10. Calculate Early Warning Score
# ============================================================

RISK_WEIGHT = 0.50
FORECAST_WEIGHT = 0.30
ANOMALY_WEIGHT = 0.20


def calculate_early_warning_score(row):

    weighted_sum = (
        row["risk_component_score"]
        * RISK_WEIGHT
    )

    available_weight = (
        RISK_WEIGHT
    )

    if row["forecast_available"]:

        weighted_sum += (
            row[
                "forecast_component_score"
            ]
            * FORECAST_WEIGHT
        )

        available_weight += (
            FORECAST_WEIGHT
        )

    weighted_sum += (
        row[
            "anomaly_component_score"
        ]
        * ANOMALY_WEIGHT
    )

    available_weight += (
        ANOMALY_WEIGHT
    )

    return (
        weighted_sum
        / available_weight
    )


master[
    "early_warning_score"
] = master.apply(
    calculate_early_warning_score,
    axis=1
)


print("=" * 74)
print("EARLY WARNING SCORE")
print("=" * 74)

print(
    "\nMinimum:",
    round(
        master[
            "early_warning_score"
        ].min(),
        2
    )
)

print(
    "Mean:",
    round(
        master[
            "early_warning_score"
        ].mean(),
        2
    )
)

print(
    "Maximum:",
    round(
        master[
            "early_warning_score"
        ].max(),
        2
    )
)

print(
    "\nMissing scores:",
    int(
        master[
            "early_warning_score"
        ].isna().sum()
    )
)

EARLY WARNING SCORE

Minimum: 0.0
Mean: 23.15
Maximum: 100.0

Missing scores: 0


## Convert score into priority

In [11]:
# ============================================================
# 11. Assign Early Warning Priority
# ============================================================

def assign_priority(score):

    if score >= 60:
        return "High"

    elif score >= 30:
        return "Medium"

    else:
        return "Low"


master[
    "early_warning_priority"
] = (
    master[
        "early_warning_score"
    ].apply(
        assign_priority
    )
)


print("=" * 74)
print("EARLY WARNING PRIORITY DISTRIBUTION")
print("=" * 74)

priority_counts = (
    master[
        "early_warning_priority"
    ]
    .value_counts()
    .reindex(
        [
            "High",
            "Medium",
            "Low"
        ]
    )
)

priority_percent = (
    master[
        "early_warning_priority"
    ]
    .value_counts(
        normalize=True
    )
    .mul(100)
    .reindex(
        [
            "High",
            "Medium",
            "Low"
        ]
    )
)


priority_summary = pd.DataFrame({
    "Records":
        priority_counts,

    "Percent":
        priority_percent.round(2)
})


display(
    priority_summary
)

EARLY WARNING PRIORITY DISTRIBUTION


,Records,Percent
early_warning_priority,,
High,141,8.39
Medium,564,33.57
Low,975,58.04


## Cross-check priority against AI signals

In [12]:
# ============================================================
# 12. Priority signal consistency audit
# ============================================================

priority_signal_audit = (
    master
    .groupby(
        "early_warning_priority"
    )
    .agg(
        records=(
            "record_id",
            "size"
        ),

        mean_warning_score=(
            "early_warning_score",
            "mean"
        ),

        mean_achievement=(
            "achievement_rate",
            "mean"
        ),

        high_risk_cases=(
            "predicted_risk",
            lambda x:
                (x == "High").sum()
        ),

        medium_risk_cases=(
            "predicted_risk",
            lambda x:
                (x == "Medium").sum()
        ),

        anomaly_alerts=(
            "predicted_anomaly",
            "sum"
        ),

        forecasts_available=(
            "forecast_available",
            "sum"
        ),

        critical_forecasts=(
            "forecast_status",
            lambda x:
                (x == "Critical").sum()
        ),

        watch_forecasts=(
            "forecast_status",
            lambda x:
                (x == "Watch").sum()
        )
    )
    .reindex(
        [
            "High",
            "Medium",
            "Low"
        ]
    )
)


priority_signal_audit[
    "mean_warning_score"
] = (
    priority_signal_audit[
        "mean_warning_score"
    ].round(2)
)

priority_signal_audit[
    "mean_achievement"
] = (
    priority_signal_audit[
        "mean_achievement"
    ].round(3)
)


print("=" * 76)
print("PRIORITY SIGNAL CONSISTENCY AUDIT")
print("=" * 76)

display(
    priority_signal_audit
)

PRIORITY SIGNAL CONSISTENCY AUDIT


,records,mean_warning_score,mean_achievement,high_risk_cases,medium_risk_cases,anomaly_alerts,forecasts_available,critical_forecasts,watch_forecasts
early_warning_priority,,,,,,,,,
High,141,73.94,0.613,139,2,28,118,41,75
Medium,564,40.33,0.688,1,562,5,461,41,414
Low,975,5.87,0.788,0,51,1,861,0,295


## Audit cross-signal boundary cases

In [13]:
# ============================================================
# 13. Audit cross-signal boundary cases
# ============================================================

boundary_cases = master[
    (
        (
            master["predicted_risk"] == "High"
        )
        &
        (
            master["early_warning_priority"] != "High"
        )
    )
    |
    (
        (
            master["predicted_risk"] == "Medium"
        )
        &
        (
            master["early_warning_priority"] == "High"
        )
    )
    |
    (
        (
            master["predicted_anomaly"] == 1
        )
        &
        (
            master["early_warning_priority"] == "Low"
        )
    )
].copy()


print("=" * 78)
print("CROSS-SIGNAL BOUNDARY CASE AUDIT")
print("=" * 78)

print(
    "\nBoundary cases:",
    len(boundary_cases)
)

display(
    boundary_cases[
        [
            "record_id",
            "reporting_date",
            "programme_name",
            "lga",
            "community",
            "predicted_risk",
            "forecast_status",
            "predicted_anomaly",
            "risk_component_score",
            "forecast_component_score",
            "anomaly_component_score",
            "early_warning_score",
            "early_warning_priority",
            "primary_driver"
        ]
    ]
    .sort_values(
        "early_warning_score",
        ascending=False
    )
)

CROSS-SIGNAL BOUNDARY CASE AUDIT

Boundary cases: 4


,record_id,reporting_date,programme_name,lga,community,predicted_risk,forecast_status,predicted_anomaly,risk_component_score,forecast_component_score,anomaly_component_score,early_warning_score,early_warning_priority,primary_driver
270,ES-005275,2026-03-01,Nutrition Support Programme,Mafa,Mafa Site 3,Medium,Watch,1,50.0,50.0,100.0,60.0,High,Elevated complaints
962,ES-006622,2026-06-01,WASH Resilience Initiative,Konduga,Konduga Site 3,Medium,Watch,1,50.0,50.0,100.0,60.0,High,Elevated complaints
1032,ES-000766,2026-06-01,Community Resilience Programme,Mafa,Mafa Site 6,High,On Track,0,100.0,0.0,0.0,50.0,Medium,Reporting delay
936,ES-003485,2026-05-01,Education Access Project,Konduga,Konduga Site 1,Low,On Track,1,0.0,0.0,100.0,20.0,Low,Elevated complaints


## Audit all High-Risk cases

In [14]:
# ============================================================
# 14. High-Risk priority coverage audit
# ============================================================

high_risk_priority_audit = (
    master[
        master[
            "predicted_risk"
        ] == "High"
    ]
    [
        [
            "record_id",
            "reporting_date",
            "programme_name",
            "lga",
            "community",
            "predicted_risk",
            "probability_high",
            "forecast_status",
            "predicted_anomaly",
            "early_warning_score",
            "early_warning_priority",
            "primary_driver",
            "secondary_driver",
            "tertiary_driver"
        ]
    ]
    .sort_values(
        "early_warning_score"
    )
)


print("=" * 78)
print("HIGH-RISK PRIORITY COVERAGE")
print("=" * 78)

print(
    "\nPredicted High-Risk cases:",
    len(high_risk_priority_audit)
)

print(
    "Assigned High Priority:",
    int(
        (
            high_risk_priority_audit[
                "early_warning_priority"
            ] == "High"
        ).sum()
    )
)

print(
    "Assigned below High Priority:",
    int(
        (
            high_risk_priority_audit[
                "early_warning_priority"
            ] != "High"
        ).sum()
    )
)

print(
    "\nLowest-scoring High-Risk cases:"
)

display(
    high_risk_priority_audit.head(10)
)

HIGH-RISK PRIORITY COVERAGE

Predicted High-Risk cases: 140
Assigned High Priority: 139
Assigned below High Priority: 1

Lowest-scoring High-Risk cases:


,record_id,reporting_date,programme_name,lga,community,predicted_risk,probability_high,forecast_status,predicted_anomaly,early_warning_score,early_warning_priority,primary_driver,secondary_driver,tertiary_driver
1032,ES-000766,2026-06-01,Community Resilience Programme,Mafa,Mafa Site 6,High,0.949144,On Track,0,50.0,Medium,Reporting delay,Access constraints,Supply delay
808,ES-003037,2026-05-01,Livelihood Recovery Initiative,Gwoza,Gwoza Site 5,High,0.773365,Watch,0,65.0,High,Supply delay,Low achievement,Low activity completion
531,ES-005276,2026-04-01,Nutrition Support Programme,Mafa,Mafa Site 3,High,0.996512,Watch,0,65.0,High,Low achievement,Lower data quality,Previous-month achievement pattern
1055,ES-002302,2026-06-01,Livelihood Recovery Initiative,Mafa,Mafa Site 6,High,0.753792,Watch,0,65.0,High,Low activity completion,Access constraints,Low achievement
520,ES-004316,2026-04-01,Education Access Project,Dikwa,Dikwa Site 3,High,0.546971,Watch,0,65.0,High,Low achievement,Elevated complaints,Access constraints
1064,ES-003646,2026-06-01,Education Access Project,Konduga,Konduga Site 6,High,0.919814,Watch,0,65.0,High,Low achievement,Supply delay,Reporting delay
1078,ES-003230,2026-06-01,Education Access Project,Maiduguri,Maiduguri Site 5,High,0.999755,Watch,0,65.0,High,Supply delay,Access constraints,Low staff availability
1087,ES-004158,2026-06-01,Education Access Project,Ngala,Ngala Site 4,High,0.970719,Watch,0,65.0,High,Low activity completion,Low achievement,Reporting delay
1088,ES-004190,2026-06-01,Education Access Project,Ngala,Ngala Site 5,High,0.970552,Watch,0,65.0,High,Low activity completion,Supply delay,Reporting delay
1122,ES-001790,2026-06-01,Livelihood Recovery Initiative,Jere,Jere Site 2,High,0.590391,Watch,0,65.0,High,Low activity completion,Low achievement,Elevated complaints


## Audit anomaly priority coverage

In [15]:
# ============================================================
# 15. Anomaly priority coverage audit
# ============================================================

anomaly_priority_audit = (
    master[
        master[
            "predicted_anomaly"
        ] == 1
    ]
    [
        [
            "record_id",
            "reporting_date",
            "programme_name",
            "lga",
            "community",
            "achievement_rate",
            "predicted_risk",
            "forecast_status",
            "anomaly_score",
            "early_warning_score",
            "early_warning_priority",
            "primary_driver"
        ]
    ]
    .sort_values(
        "early_warning_score"
    )
)


print("=" * 78)
print("ANOMALY PRIORITY COVERAGE")
print("=" * 78)

print(
    "\nDetected anomalies:",
    len(anomaly_priority_audit)
)

print(
    "\nPriority distribution:"
)

print(
    anomaly_priority_audit[
        "early_warning_priority"
    ].value_counts()
)

print(
    "\nLowest-priority anomaly cases:"
)

display(
    anomaly_priority_audit.head(10)
)

ANOMALY PRIORITY COVERAGE

Detected anomalies: 34

Priority distribution:
early_warning_priority
High      28
Medium     5
Low        1
Name: count, dtype: int64

Lowest-priority anomaly cases:


,record_id,reporting_date,programme_name,lga,community,achievement_rate,predicted_risk,forecast_status,anomaly_score,early_warning_score,early_warning_priority,primary_driver
936,ES-003485,2026-05-01,Education Access Project,Konduga,Konduga Site 1,0.9592,Low,On Track,0.095942,20.0,Low,Elevated complaints
1434,ES-004031,2026-07-01,Education Access Project,Bama,Bama Site 6,0.7826,Medium,On Track,0.035653,45.0,Medium,Elevated complaints
1318,ES-004735,2026-07-01,Nutrition Support Programme,Maiduguri,Maiduguri Site 4,0.7874,Medium,On Track,0.044723,45.0,Medium,Elevated complaints
944,ES-002301,2026-05-01,Livelihood Recovery Initiative,Mafa,Mafa Site 6,0.8474,Medium,On Track,0.044561,45.0,Medium,Elevated complaints
947,ES-004989,2026-05-01,Nutrition Support Programme,Jere,Jere Site 6,0.7989,Medium,On Track,0.031203,45.0,Medium,Elevated complaints
1156,ES-004766,2026-06-01,Nutrition Support Programme,Maiduguri,Maiduguri Site 5,0.8225,Medium,On Track,0.051061,45.0,Medium,Elevated complaints
270,ES-005275,2026-03-01,Nutrition Support Programme,Mafa,Mafa Site 3,0.7937,Medium,Watch,0.016238,60.0,High,Elevated complaints
962,ES-006622,2026-06-01,WASH Resilience Initiative,Konduga,Konduga Site 3,0.7082,Medium,Watch,0.011588,60.0,High,Elevated complaints
306,ES-003003,2026-03-01,Livelihood Recovery Initiative,Gwoza,Gwoza Site 4,0.8788,High,On Track,0.106902,70.0,High,Supply delay
471,ES-002619,2026-03-01,Livelihood Recovery Initiative,Ngala,Ngala Site 4,0.7803,High,On Track,0.008170,70.0,High,Supply delay


## Add signal agreement and alert strength

In [16]:
# ============================================================
# 16. Calculate signal agreement
# ============================================================

master[
    "risk_warning_signal"
] = (
    master[
        "predicted_risk"
    ].isin(
        ["Medium", "High"]
    )
)

master[
    "forecast_warning_signal"
] = (
    master[
        "forecast_status"
    ].isin(
        ["Watch", "Critical"]
    )
)

master[
    "anomaly_warning_signal"
] = (
    master[
        "predicted_anomaly"
    ] == 1
)


master[
    "warning_signal_count"
] = (
    master[
        [
            "risk_warning_signal",
            "forecast_warning_signal",
            "anomaly_warning_signal"
        ]
    ]
    .sum(axis=1)
)


def assign_signal_agreement(row):

    available_signals = (
        3
        if row["forecast_available"]
        else 2
    )

    warning_count = int(
        row[
            "warning_signal_count"
        ]
    )

    if warning_count == available_signals:
        return "Strong Agreement"

    elif warning_count >= 2:
        return "Multiple Signals"

    elif warning_count == 1:
        return "Single Signal"

    else:
        return "No Warning Signal"


master[
    "signal_agreement"
] = master.apply(
    assign_signal_agreement,
    axis=1
)


print("=" * 76)
print("EARLY WARNING SIGNAL AGREEMENT")
print("=" * 76)

print(
    "\nWarning signal count:"
)

print(
    master[
        "warning_signal_count"
    ].value_counts().sort_index()
)

print(
    "\nSignal agreement:"
)

print(
    master[
        "signal_agreement"
    ].value_counts()
)

EARLY WARNING SIGNAL AGREEMENT

Warning signal count:
warning_signal_count
0    628
1    473
2    555
3     24
Name: count, dtype: int64

Signal agreement:
signal_agreement
No Warning Signal    628
Multiple Signals     553
Single Signal        473
Strong Agreement      26
Name: count, dtype: int64


## Create operational priority message

In [17]:
# ============================================================
# 17. Create operational priority message
# ============================================================

def build_priority_message(row):

    signals = [
        f"Risk: {row['predicted_risk']}"
    ]

    if row["forecast_available"]:
        signals.append(
            f"Forecast: {row['forecast_status']}"
        )
    else:
        signals.append(
            "Forecast: Not available"
        )

    signals.append(
        "Anomaly: "
        + (
            "Alert"
            if row["predicted_anomaly"] == 1
            else "No alert"
        )
    )

    return " | ".join(signals)


master[
    "priority_message"
] = master.apply(
    build_priority_message,
    axis=1
)


print("=" * 78)
print("EARLYSIGNAL AI — SAMPLE PRIORITY ALERTS")
print("=" * 78)

sample_alerts = (
    master[
        master[
            "early_warning_priority"
        ] == "High"
    ]
    .sort_values(
        [
            "early_warning_score",
            "probability_high"
        ],
        ascending=False
    )
    .head(10)
)


display(
    sample_alerts[
        [
            "record_id",
            "programme_name",
            "lga",
            "community",
            "early_warning_priority",
            "early_warning_score",
            "priority_message",
            "signal_agreement",
            "primary_driver",
            "secondary_driver",
            "tertiary_driver"
        ]
    ]
)

EARLYSIGNAL AI — SAMPLE PRIORITY ALERTS


,record_id,programme_name,lga,community,early_warning_priority,early_warning_score,priority_message,signal_agreement,primary_driver,secondary_driver,tertiary_driver
1482,ES-002880,Livelihood Recovery Initiative,Dikwa,Dikwa Site 6,High,100.0,Risk: High | Forecast: Not available | Anomaly...,Strong Agreement,Supply delay,Low achievement,Low activity completion
1651,ES-005440,Nutrition Support Programme,Bama,Bama Site 2,High,100.0,Risk: High | Forecast: Not available | Anomaly...,Strong Agreement,Reporting delay,Low activity completion,Lower data quality
676,ES-006044,Nutrition Support Programme,Gwoza,Gwoza Site 3,High,85.0,Risk: High | Forecast: Watch | Anomaly: Alert,Strong Agreement,Supply delay,Reporting delay,Low achievement
160,ES-002426,Livelihood Recovery Initiative,Bama,Bama Site 4,High,85.0,Risk: High | Forecast: Watch | Anomaly: Alert,Strong Agreement,Supply delay,Low achievement,Low activity completion
1213,ES-007231,WASH Resilience Initiative,Ngala,Ngala Site 4,High,85.0,Risk: High | Forecast: Watch | Anomaly: Alert,Strong Agreement,Supply delay,Access constraints,Reporting delay
680,ES-006716,WASH Resilience Initiative,Konduga,Konduga Site 6,High,85.0,Risk: High | Forecast: Watch | Anomaly: Alert,Strong Agreement,Low achievement,Reporting delay,Low activity completion
354,ES-002363,Livelihood Recovery Initiative,Bama,Bama Site 2,High,85.0,Risk: High | Forecast: Watch | Anomaly: Alert,Strong Agreement,Supply delay,Reporting delay,Low activity completion
485,ES-002460,Livelihood Recovery Initiative,Bama,Bama Site 5,High,85.0,Risk: High | Forecast: Watch | Anomaly: Alert,Strong Agreement,Elevated complaints,Reporting delay,Supply delay
369,ES-002331,Livelihood Recovery Initiative,Bama,Bama Site 1,High,85.0,Risk: High | Forecast: Watch | Anomaly: Alert,Strong Agreement,Supply delay,Reporting delay,Low achievement
304,ES-002299,Livelihood Recovery Initiative,Mafa,Mafa Site 6,High,85.0,Risk: High | Forecast: Watch | Anomaly: Alert,Strong Agreement,Supply delay,Low activity completion,Access constraints


## Add forecast coverage context and review status

In [18]:
# ============================================================
# 18. Add forecast coverage context and review status
# ============================================================

master["forecast_context"] = np.where(
    master["forecast_available"],
    "Held-out forecast available",
    "Forecast unavailable for final reporting month"
)


def assign_review_status(row):

    if row["early_warning_priority"] == "High":
        return "Priority human review"

    elif row["early_warning_priority"] == "Medium":
        return "Monitor and review"

    else:
        return "Routine monitoring"


master["review_status"] = master.apply(
    assign_review_status,
    axis=1
)


print("=" * 76)
print("HUMAN-REVIEW STATUS")
print("=" * 76)

print(
    "\nReview status distribution:"
)

print(
    master[
        "review_status"
    ].value_counts()
)

print(
    "\nForecast context:"
)

print(
    master[
        "forecast_context"
    ].value_counts()
)

HUMAN-REVIEW STATUS

Review status distribution:
review_status
Routine monitoring       975
Monitor and review       564
Priority human review    141
Name: count, dtype: int64

Forecast context:
forecast_context
Held-out forecast available                       1440
Forecast unavailable for final reporting month     240
Name: count, dtype: int64


## Create consolidated investigation guidance

In [19]:
# ============================================================
# 19. Create consolidated investigation guidance
# ============================================================

def build_integrated_guidance(row):

    actions = []

    existing_guidance = row[
        "recommended_investigation"
    ]

    if pd.notna(existing_guidance):

        for action in str(
            existing_guidance
        ).split(" | "):

            action = action.strip()

            if (
                action
                and action not in actions
            ):
                actions.append(action)

    if (
        row["predicted_anomaly"] == 1
    ):

        anomaly_action = (
            "Validate the unusual operational pattern "
            "against source data and field context."
        )

        if anomaly_action not in actions:
            actions.append(
                anomaly_action
            )

    if (
        row["forecast_status"]
        == "Critical"
    ):

        forecast_action = (
            "Review the projected next-period "
            "underachievement and identify preventive actions."
        )

        if forecast_action not in actions:
            actions.append(
                forecast_action
            )

    return " | ".join(actions)


master[
    "integrated_investigation_guidance"
] = master.apply(
    build_integrated_guidance,
    axis=1
)


print("=" * 76)
print("INTEGRATED INVESTIGATION GUIDANCE")
print("=" * 76)

print(
    "\nRecords with guidance:",
    int(
        master[
            "integrated_investigation_guidance"
        ].str.len().gt(0).sum()
    )
)

print(
    "High-Priority records with guidance:",
    int(
        master.loc[
            master[
                "early_warning_priority"
            ] == "High",
            "integrated_investigation_guidance"
        ].str.len().gt(0).sum()
    )
)

INTEGRATED INVESTIGATION GUIDANCE

Records with guidance: 1679
High-Priority records with guidance: 141


## Create final ranked alert table

In [20]:
# ============================================================
# 20. Create final ranked EarlySignal alert table
# ============================================================

priority_rank_map = {
    "High": 3,
    "Medium": 2,
    "Low": 1
}

master[
    "priority_rank"
] = master[
    "early_warning_priority"
].map(
    priority_rank_map
)


ranked_alerts = (
    master
    .sort_values(
        [
            "priority_rank",
            "early_warning_score",
            "probability_high",
            "anomaly_score"
        ],
        ascending=[
            False,
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)


ranked_alerts[
    "alert_rank"
] = (
    np.arange(
        1,
        len(ranked_alerts) + 1
    )
)


print("=" * 78)
print("EARLYSIGNAL AI — RANKED EARLY WARNING ALERTS")
print("=" * 78)

display(
    ranked_alerts[
        [
            "alert_rank",
            "record_id",
            "reporting_date",
            "programme_name",
            "lga",
            "community",
            "early_warning_priority",
            "early_warning_score",
            "predicted_risk",
            "forecast_status",
            "predicted_anomaly",
            "signal_agreement",
            "primary_driver",
            "review_status"
        ]
    ].head(20)
)

EARLYSIGNAL AI — RANKED EARLY WARNING ALERTS


,alert_rank,record_id,reporting_date,programme_name,lga,community,early_warning_priority,early_warning_score,predicted_risk,forecast_status,predicted_anomaly,signal_agreement,primary_driver,review_status
0,1,ES-002880,2026-08-01,Livelihood Recovery Initiative,Dikwa,Dikwa Site 6,High,100.0,High,NaN,1,Strong Agreement,Supply delay,Priority human review
1,2,ES-005440,2026-08-01,Nutrition Support Programme,Bama,Bama Site 2,High,100.0,High,NaN,1,Strong Agreement,Reporting delay,Priority human review
2,3,ES-006044,2026-04-01,Nutrition Support Programme,Gwoza,Gwoza Site 3,High,85.0,High,Watch,1,Strong Agreement,Supply delay,Priority human review
3,4,ES-002426,2026-02-01,Livelihood Recovery Initiative,Bama,Bama Site 4,High,85.0,High,Watch,1,Strong Agreement,Supply delay,Priority human review
4,5,ES-007231,2026-07-01,WASH Resilience Initiative,Ngala,Ngala Site 4,High,85.0,High,Watch,1,Strong Agreement,Supply delay,Priority human review
5,6,ES-006716,2026-04-01,WASH Resilience Initiative,Konduga,Konduga Site 6,High,85.0,High,Watch,1,Strong Agreement,Low achievement,Priority human review
6,7,ES-002363,2026-03-01,Livelihood Recovery Initiative,Bama,Bama Site 2,High,85.0,High,Watch,1,Strong Agreement,Supply delay,Priority human review
7,8,ES-002460,2026-04-01,Livelihood Recovery Initiative,Bama,Bama Site 5,High,85.0,High,Watch,1,Strong Agreement,Elevated complaints,Priority human review
8,9,ES-002331,2026-03-01,Livelihood Recovery Initiative,Bama,Bama Site 1,High,85.0,High,Watch,1,Strong Agreement,Supply delay,Priority human review
9,10,ES-002299,2026-03-01,Livelihood Recovery Initiative,Mafa,Mafa Site 6,High,85.0,High,Watch,1,Strong Agreement,Supply delay,Priority human review


## Final engine integrity audit

In [21]:
# ============================================================
# 21. Final Early Warning Engine integrity audit
# ============================================================

high_priority = (
    ranked_alerts[
        ranked_alerts[
            "early_warning_priority"
        ] == "High"
    ]
)

integrity_checks = {
    "1680 records retained":
        len(ranked_alerts) == 1680,

    "Unique record IDs":
        ranked_alerts[
            "record_id"
        ].is_unique,

    "No missing priority scores":
        ranked_alerts[
            "early_warning_score"
        ].notna().all(),

    "No missing priorities":
        ranked_alerts[
            "early_warning_priority"
        ].notna().all(),

    "No missing primary explanations":
        ranked_alerts[
            "primary_driver"
        ].notna().all(),

    "All High-Priority cases have guidance":
        high_priority[
            "integrated_investigation_guidance"
        ].str.len().gt(0).all(),

    "All High-Priority cases marked for priority review":
        (
            high_priority[
                "review_status"
            ]
            == "Priority human review"
        ).all(),

    "Forecast coverage correctly retained":
        int(
            ranked_alerts[
                "forecast_available"
            ].sum()
        ) == 1440
}


print("=" * 78)
print("EARLY WARNING ENGINE — FINAL INTEGRITY AUDIT")
print("=" * 78)

for check, passed in integrity_checks.items():

    print(
        f"{check}: "
        f"{'PASSED' if passed else 'FAILED'}"
    )


all_checks_passed = all(
    integrity_checks.values()
)

print(
    "\nOverall integrity audit:",
    (
        "PASSED"
        if all_checks_passed
        else "CHECK REQUIRED"
    )
)

EARLY WARNING ENGINE — FINAL INTEGRITY AUDIT
1680 records retained: PASSED
Unique record IDs: PASSED
No missing priority scores: PASSED
No missing priorities: PASSED
No missing primary explanations: PASSED
All High-Priority cases have guidance: PASSED
All High-Priority cases marked for priority review: PASSED
Forecast coverage correctly retained: PASSED

Overall integrity audit: PASSED


## Save final Priority Engine output

In [22]:
# ============================================================
# 22. Save final Early Warning Priority Engine output
# ============================================================

FINAL_OUTPUT_PATH = (
    OUTPUT_DIR
    / "early_warning_priority_results.csv"
)


output_columns = [
    "alert_rank",
    "record_id",
    "reporting_date",
    "programme_name",
    "sector",
    "state",
    "lga",
    "community",
    "achievement_rate",
    "predicted_risk",
    "probability_high",
    "forecast_available",
    "predicted_next_achievement_rate",
    "forecast_status",
    "predicted_anomaly",
    "anomaly_score",
    "risk_component_score",
    "forecast_component_score",
    "anomaly_component_score",
    "early_warning_score",
    "early_warning_priority",
    "warning_signal_count",
    "signal_agreement",
    "primary_driver",
    "secondary_driver",
    "tertiary_driver",
    "strongest_protective_signal",
    "priority_message",
    "integrated_investigation_guidance",
    "review_status",
    "forecast_context",
    "decision_use"
]


ranked_alerts[
    output_columns
].to_csv(
    FINAL_OUTPUT_PATH,
    index=False
)


saved_results = pd.read_csv(
    FINAL_OUTPUT_PATH
)


print("=" * 78)
print("EARLY WARNING PRIORITY OUTPUT SAVED")
print("=" * 78)

print("\nSaved to:")
print(
    FINAL_OUTPUT_PATH
)

print(
    "\nFile exists:",
    FINAL_OUTPUT_PATH.exists()
)

print(
    "Rows saved:",
    len(saved_results)
)

print(
    "Columns saved:",
    len(saved_results.columns)
)

print(
    "Duplicate record IDs:",
    int(
        saved_results[
            "record_id"
        ].duplicated().sum()
    )
)

EARLY WARNING PRIORITY OUTPUT SAVED

Saved to:
C:\Users\Ezekiel Mbaya\myenv\EarlySignal_AI\outputs\early_warning_priority_results.csv

File exists: True
Rows saved: 1680
Columns saved: 32
Duplicate record IDs: 0


## Final Notebook 07 summary

In [23]:
# ============================================================
# 23. Final Early Warning Priority Engine summary
# ============================================================

priority_counts = (
    ranked_alerts[
        "early_warning_priority"
    ].value_counts()
)

high_count = int(
    priority_counts.get(
        "High",
        0
    )
)

medium_count = int(
    priority_counts.get(
        "Medium",
        0
    )
)

low_count = int(
    priority_counts.get(
        "Low",
        0
    )
)


print("=" * 80)
print("EARLYSIGNAL AI — EARLY WARNING PRIORITY ENGINE SUMMARY")
print("=" * 80)

print(
    f"Integrated monitoring records: "
    f"{len(ranked_alerts):,}"
)

print(
    f"Records with forecast evidence: "
    f"{int(ranked_alerts['forecast_available'].sum()):,}"
)

print(
    f"Records without held-out forecast evidence: "
    f"{int((~ranked_alerts['forecast_available']).sum()):,}"
)

print(
    "\nComponent weights:"
)

print(
    "Risk Intelligence:     50%"
)

print(
    "Performance Forecast: 30%"
)

print(
    "Anomaly Detection:    20%"
)

print(
    "\nPriority thresholds:"
)

print(
    "High:   score >= 60"
)

print(
    "Medium: score >= 30 and < 60"
)

print(
    "Low:    score < 30"
)

print(
    "\nPriority distribution:"
)

print(
    f"High:   {high_count:,} "
    f"({high_count / len(ranked_alerts):.2%})"
)

print(
    f"Medium: {medium_count:,} "
    f"({medium_count / len(ranked_alerts):.2%})"
)

print(
    f"Low:    {low_count:,} "
    f"({low_count / len(ranked_alerts):.2%})"
)

print(
    "\nSignal agreement:"
)

print(
    ranked_alerts[
        "signal_agreement"
    ].value_counts()
)

print(
    "\nScoring note: missing forecast evidence is not "
    "treated as a zero-risk forecast. Available component "
    "weights are renormalized."
)

print(
    "Interpretation note: the Early Warning Score is a "
    "transparent prototype prioritization index, not a "
    "probability of programme failure."
)

print(
    "Validation note: model performance and priority-engine "
    "behaviour were evaluated using synthetic monitoring data."
)

print(
    "Decision-use note: alerts identify records recommended "
    "for human review and do not automate operational decisions."
)

print(
    "\nFinal integrity audit:",
    (
        "PASSED"
        if all_checks_passed
        else "CHECK REQUIRED"
    )
)

print(
    "\nNotebook 07 complete."
)

EARLYSIGNAL AI — EARLY WARNING PRIORITY ENGINE SUMMARY
Integrated monitoring records: 1,680
Records with forecast evidence: 1,440
Records without held-out forecast evidence: 240

Component weights:
Risk Intelligence:     50%
Performance Forecast: 30%
Anomaly Detection:    20%

Priority thresholds:
High:   score >= 60
Medium: score >= 30 and < 60
Low:    score < 30

Priority distribution:
High:   141 (8.39%)
Medium: 564 (33.57%)
Low:    975 (58.04%)

Signal agreement:
signal_agreement
No Warning Signal    628
Multiple Signals     553
Single Signal        473
Strong Agreement      26
Name: count, dtype: int64

Scoring note: missing forecast evidence is not treated as a zero-risk forecast. Available component weights are renormalized.
Interpretation note: the Early Warning Score is a transparent prototype prioritization index, not a probability of programme failure.
Validation note: model performance and priority-engine behaviour were evaluated using synthetic monitoring data.
Decision-us